In [1]:
# ===== Phase 4 · Batch A: pre-checks P1–P3 (read-only) =====
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Works whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import baselines, features, validation
from config import DATE_COL, TRAIN_PATH, VALID_PATH

raw = pd.read_csv(TRAIN_PATH)
vraw = pd.read_csv(VALID_PATH)

# Split RAW rows first, then clean each side with values learned from its own
# training side, exactly as run_split does. (market_index is filled by
# same-date mean within each file, so the learned dict doesn't change it.)
dec_tr_raw, aug_raw = validation.time_split(raw, validation.DECISION_FOLD)
main_tr_raw, so_raw = validation.time_split(raw, validation.MAIN_FOLD)

L_dec, L_main, L_full = (features.fit_cleaning(x) for x in (dec_tr_raw, main_tr_raw, raw))
windows = {
    "dec-train Jan-Jul":  features.clean(dec_tr_raw, L_dec),
    "Aug":                features.clean(aug_raw, L_dec),
    "main-train Jan-Aug": features.clean(main_tr_raw, L_main),
    "Sep-Oct":            features.clean(so_raw, L_main),
    "full-train Jan-Oct": features.clean(raw, L_full),     # what the final model sees
    "valid Nov-Dec":      features.clean(vraw, L_full),
}

# Guard: row counts must match the handoff, otherwise stop
expected = {"dec-train Jan-Jul": 33718, "Aug": 4759, "main-train Jan-Aug": 38477,
            "Sep-Oct": 9523, "full-train Jan-Oct": 48000, "valid Nov-Dec": 12000}
for k, n in expected.items():
    assert len(windows[k]) == n, f"{k}: {len(windows[k])} rows, expected {n}"
print("Row counts match the handoff.\n")

# ---------- P1: market_index range ----------
qs = [0, .01, .05, .5, .95, .99, 1]
names = ["min", "p1", "p5", "p50", "p95", "p99", "max"]

def mi_stats(df):
    s = df["market_index"]
    out = pd.Series(s.quantile(qs).to_numpy(), index=names)
    daily = df.groupby(DATE_COL)["market_index"].mean()
    out["daily_min"], out["daily_max"] = daily.min(), daily.max()
    out["n_missing_after_clean"] = s.isna().sum()
    return out

print("P1a. market_index per window (cleaned):")
print(pd.DataFrame({k: mi_stats(d) for k, d in windows.items()}).T.round(4).to_string(), "\n")

def outside(later, train):
    lo, hi = train["market_index"].min(), train["market_index"].max()
    p5 = train["market_index"].quantile(.05)
    s = later["market_index"]
    return {"train_min": lo, "train_max": hi, "later_min": s.min(), "later_max": s.max(),
            "n_below_min": int((s < lo).sum()), "n_above_max": int((s > hi).sum()),
            "pct_outside": 100 * ((s < lo) | (s > hi)).mean(),
            "pct_below_train_p5": 100 * (s < p5).mean()}   # sparse-region check

pairs = {"Aug vs dec-train":        ("Aug", "dec-train Jan-Jul"),
         "Sep-Oct vs main-train":   ("Sep-Oct", "main-train Jan-Aug"),
         "Nov-Dec vs full-train":   ("valid Nov-Dec", "full-train Jan-Oct")}
print("P1b. Later rows vs their training range:")
print(pd.DataFrame({k: outside(windows[a], windows[b]) for k, (a, b) in pairs.items()})
      .T.round(4).to_string(), "\n")

# December daily means (what the December chart will be fed, D17)
v = windows["valid Nov-Dec"]
dec_daily = v[v[DATE_COL].dt.month == 12].groupby(DATE_COL)["market_index"].mean()
ft = windows["full-train Jan-Oct"]["market_index"]
print(f"P1c. December daily mean market_index: min {dec_daily.min():.4f}, max {dec_daily.max():.4f}, "
      f"days {len(dec_daily)}; full-train range {ft.min():.4f}-{ft.max():.4f}\n")

allmi = pd.concat([windows["full-train Jan-Oct"], v])
print("P1d. Median market_index by month (Jan-Dec):")
print(allmi.groupby(allmi[DATE_COL].dt.strftime("%Y-%m"))["market_index"].median().round(4).to_string(), "\n")

# ---------- P2: quote_signal missing values ----------
print("P2. quote_signal missing values (raw; clean() doesn't touch this column):")
for name, d in [("train Jan-Oct", raw), ("validation Nov-Dec", vraw)]:
    s = pd.to_numeric(d["quote_signal"], errors="coerce")
    print(f"  {name}: dtype {d['quote_signal'].dtype}, rows {len(s)}, missing {s.isna().sum()} "
          f"({100*s.isna().mean():.2f}%), non-finite {int((~np.isfinite(s.dropna())).sum())}")
both = pd.concat([raw.assign(f="train"), vraw.assign(f="valid")])
both["month"] = pd.to_datetime(both[DATE_COL]).dt.strftime("%Y-%m")
print("  missing by month:", both.groupby("month")["quote_signal"].apply(lambda s: int(s.isna().sum())).to_dict(), "\n")

# ---------- P3: smallest price-table cell per fold ----------
# Counted on cleaned training rows BEFORE dropping errors (the table is fitted on these).
def cells(train):
    band = features.distance_band(train["distance"]).astype(str).rename("band")
    flags = baselines.flag_label_errors(train, baselines.fit_price_table(train))
    g = pd.DataFrame({"rows": 1, "flagged": flags["is_error"].astype(int)}) \
          .groupby([train["equipment"], band]).sum()
    return g

for k in ["dec-train Jan-Jul", "main-train Jan-Aug", "full-train Jan-Oct"]:
    g = cells(windows[k])
    smallest = g["rows"].idxmin()
    print(f"P3. {k}: {len(g)} cells; smallest {smallest} = {g.loc[smallest, 'rows']} rows "
          f"({g.loc[smallest, 'flagged']} flagged)")
    print(g["rows"].unstack("band").to_string(), "\n")

Row counts match the handoff.

P1a. market_index per window (cleaned):
                       min      p1      p5     p50     p95     p99     max  daily_min  daily_max  n_missing_after_clean
dec-train Jan-Jul   0.7325  0.8092  0.8719  1.1608  1.3775  1.4126  1.4678     0.8019     1.4016                    0.0
Aug                 0.7627  0.8252  0.8556  0.9825  1.0990  1.1276  1.1780     0.8434     1.1060                    0.0
main-train Jan-Aug  0.7325  0.8111  0.8671  1.1200  1.3735  1.4103  1.4678     0.8019     1.4016                    0.0
Sep-Oct             0.6764  0.7545  0.7897  0.9282  1.0525  1.0816  1.1411     0.7666     1.0684                    0.0
full-train Jan-Oct  0.6764  0.7831  0.8363  1.0558  1.3659  1.4070  1.4678     0.7666     1.4016                    0.0
valid Nov-Dec       0.7236  0.7912  0.8149  0.9226  1.0406  1.0656  1.0990     0.8222     1.0445                    0.0 

P1b. Later rows vs their training range:
                       train_min  train_max  l

In [2]:
# ===== Batch B · cell 1: check the installed scikit-learn, don't assume =====
import inspect
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor

print("scikit-learn", sklearn.__version__)
sig = inspect.signature(HistGradientBoostingRegressor)
for p in ["early_stopping", "validation_fraction", "n_iter_no_change", "max_iter",
          "learning_rate", "max_leaf_nodes", "min_samples_leaf", "categorical_features", "random_state"]:
    print(f"  default {p} = {sig.parameters[p].default!r}")

doc = HistGradientBoostingRegressor.__doc__
i = doc.find("early_stopping :")
print("\n--- docs: early_stopping ---\n", doc[i:i + 500])

tiny = HistGradientBoostingRegressor(max_iter=5).fit(np.arange(20.).reshape(-1, 1), np.arange(20.))
print("\nhas feature_importances_ after fit:", hasattr(tiny, "feature_importances_"))

scikit-learn 1.9.1
  default early_stopping = 'auto'
  default validation_fraction = 0.1
  default n_iter_no_change = 10
  default max_iter = 100
  default learning_rate = 0.1
  default max_leaf_nodes = 31
  default min_samples_leaf = 20
  default categorical_features = 'from_dtype'
  default random_state = None

--- docs: early_stopping ---
 early_stopping : 'auto' or bool, default='auto'
        If 'auto', early stopping is enabled if the sample size is larger than
        10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stopping
        is enabled, otherwise early stopping is disabled.

        .. versionadded:: 0.23

    scoring : str or callable or None, default='loss'
        Scoring method to use for early stopping. Only used if `early_stopping`
        is enabled. Options:

        - str: see :ref:`scoring_

has feature_importances_ after fit: False


In [3]:
# ===== Batch B · cell 2: decision fold (Jan-Jul -> Aug) =====
import importlib, time
from functools import partial
import features, baselines, validation          # reload in dependency order, so file edits are picked up
for m in (features, baselines, validation):
    importlib.reload(m)
import models, experiments
importlib.reload(models); importlib.reload(experiments)

BASE = "base(distance,weight,4 coords,market_index,equipment one-hot)"
runs = {   # key: (model factory, target, features, settings) -> written to the log
    "B1":           (baselines.SegmentMedianRpm, "rpm (median)", "equipment+band", "median table"),
    "S1":           (models.MarketAdjustedRpm, "log_rpm", "equipment+band+market_index", "OLS slope"),
    "HGB log_rpm":  (partial(models.HGBRate, target="log_rpm"), "log_rpm", BASE, "defaults; early_stopping=False; seed 42"),
    "HGB log_rate": (partial(models.HGBRate, target="log_rate"), "log_rate", BASE, "defaults; early_stopping=False; seed 42"),
}
results, rows = {}, []
for key, (make, tgt, feats, sett) in runs.items():
    t0 = time.perf_counter()
    results[key] = validation.run_fold(raw, validation.DECISION_FOLD, make)
    rows.append(experiments.experiment_row(results[key], tgt, feats, sett))
    r = results[key]
    print(f"{key:13s} fit_rows {r['fit_rows']:,}  dropped {r['dropped']}  test_rows {r['test_rows']:,}  "
          f"cleaning_rows {r['cleaning_rows']:,}  {time.perf_counter() - t0:.1f}s")

log = experiments.log_experiments(rows)

# Check 1: B1 must reproduce Phase 3 exactly, which proves the pipeline is unchanged
assert round(results["B1"]["scores"]["MAE"], 2) == 136.60, results["B1"]["scores"]["MAE"]
print("\nB1 reproduces the Phase 3 bar (MAE 136.60).")

# Side by side, with differences vs B1 (decision 7 rule: MAE gain >= $1, MAPE rise <= 0.10 pp)
tab = validation.results_table(list(results.values()))
b1 = tab[tab["model"].str.startswith("B1")].set_index("scope")
tab["dMAE_vs_B1"] = tab["MAE"] - tab["scope"].map(b1["MAE"])
tab["dMAPE_pp_vs_B1"] = tab["MAPE_%"] - tab["scope"].map(b1["MAPE_%"])
print(tab.drop(columns=["fold", "train_label_errors"]).round(2).to_string(index=False))
print("\nLog file now:\n", log.round(2).to_string(index=False))

B1            fit_rows 33,253  dropped 465  test_rows 4,759  cleaning_rows 33,718  0.5s
S1            fit_rows 33,253  dropped 465  test_rows 4,759  cleaning_rows 33,718  0.4s
HGB log_rpm   fit_rows 33,253  dropped 465  test_rows 4,759  cleaning_rows 33,718  4.2s
HGB log_rate  fit_rows 33,253  dropped 465  test_rows 4,759  cleaning_rows 33,718  2.5s

B1 reproduces the Phase 3 bar (MAE 136.60).
                            model      scope  rows    MAE  MAPE_%   RMSE   bias  dMAE_vs_B1  dMAPE_pp_vs_B1
B1 median rpm by equipment x band        all  4759 136.60    6.39 621.99   9.68        0.00            0.00
B1 median rpm by equipment x band clean-only  4691  87.25    3.98 122.94  38.14        0.00            0.00
             S1 B1 x market slope        all  4759 134.79    6.40 624.17 -67.88       -1.82            0.01
             S1 B1 x market slope clean-only  4691  85.38    4.07 113.38 -39.59       -1.87            0.09
                      HGB log_rpm        all  4759 110.04    4.

In [4]:
# ===== Batch B · cell 3: prove predict() never reads posted_rate =====
from config import TARGET
tr_raw, te_raw = validation.time_split(raw, validation.DECISION_FOLD)
learned = features.fit_cleaning(tr_raw)
tr, te = features.clean(tr_raw, learned), features.clean(te_raw, learned)
tr_fit, _ = baselines.drop_label_errors(tr)             # same rule as run_split
te_blind = te.drop(columns=[TARGET])                    # no answers available at all

for key, (make, *_ ) in runs.items():
    m = make().fit(tr_fit)
    p_with, p_blind = m.predict(te), m.predict(te_blind)
    same_as_run = np.allclose(p_with, results[key]["holdout"]["pred"].to_numpy())
    print(f"{key:13s} blind == with target: {np.array_equal(p_with, p_blind)}   "
          f"matches run_fold: {same_as_run}   n={len(p_blind)}  min ${p_blind.min():.0f}  max ${p_blind.max():.0f}")
    if key == "S1":
        print(f"{'':13s} slope {m.slope_:+.4f} per market_index unit, intercept {m.intercept_:+.4f}, "
              f"training market mean {m.market_mean_:.4f}")
    if key.startswith("HGB"):
        print(f"{'':13s} boosting iterations used: {m.model_.n_iter_}")

B1            blind == with target: True   matches run_fold: True   n=4759  min $184  max $6725
S1            blind == with target: True   matches run_fold: True   n=4759  min $174  max $6584
              slope +0.2041 per market_index unit, intercept -0.0008, training market mean 1.1423
HGB log_rpm   blind == with target: True   matches run_fold: True   n=4759  min $196  max $6450
              boosting iterations used: 100
HGB log_rate  blind == with target: True   matches run_fold: True   n=4759  min $209  max $6434
              boosting iterations used: 100


In [5]:
# ===== Batch C · cell 1 =====
import pandas as pd
from config import DATE_COL, TARGET
importlib.reload(models)                       # pick up the edited HGBRate

# 1) How would early stopping have held out its rows? (closes the Batch B wording check)
i = doc.find("validation_fraction :")
print(doc[i:i + 450], "\n")

# 2) + 3) Two experiments vs the current candidate (HGB log_rpm, base features)
SET = "defaults; early_stopping=False; seed 42"
c_runs = {
    "HGB +quote_signal": (partial(models.HGBRate, target="log_rpm", extra=("quote_signal",)),
                          "log_rpm", BASE + " + quote_signal", SET),
    "HGB smear":         (partial(models.HGBRate, target="log_rpm", smearing=True),
                          "log_rpm", BASE, SET + "; Duan smearing"),
}
c_rows = []
for key, (make, tgt, feats, sett) in c_runs.items():
    results[key] = validation.run_fold(raw, validation.DECISION_FOLD, make)
    c_rows.append(experiments.experiment_row(results[key], tgt, feats, sett))
log = experiments.log_experiments(c_rows)

# Rule (decision 7) vs the candidate: added complexity must cut MAE >= $1, MAPE rise <= 0.10 pp
ref = results["HGB log_rpm"]["model"]
cmp = validation.results_table([results[k] for k in ["HGB log_rpm", *c_runs]])
r0 = cmp[cmp["model"] == ref].set_index("scope")
cmp["dMAE_vs_HGB"] = cmp["MAE"] - cmp["scope"].map(r0["MAE"])
cmp["dMAPE_pp_vs_HGB"] = cmp["MAPE_%"] - cmp["scope"].map(r0["MAPE_%"])
print(cmp.drop(columns=["fold", "train_label_errors"]).round(2).to_string(index=False))

# Smearing factor vs theory: for small normal log errors, factor ~ exp(SD^2 / 2)
m_sm = models.HGBRate(target="log_rpm", smearing=True).fit(tr_fit)
print(f"\nDuan factor {m_sm.smear_:.5f} | training log-residual SD {m_sm.train_resid_sd_:.4f} | "
      f"exp(SD^2/2) = {np.exp(m_sm.train_resid_sd_ ** 2 / 2):.5f}")
print(f"Log rows: {len(log)}")

validation_fraction : int or float or None, default=0.1
        Proportion (or absolute size) of training data to set aside as
        validation data for early stopping. If None, early stopping is done on
        the training data.
        The value is ignored if either early stopping is not performed, e.g.
        `early_stopping=False`, or if `X_val` and `y_val` are passed to fit.
    n_iter_no_change : int, default=10
        Used to determin 

                    model      scope  rows    MAE  MAPE_%   RMSE   bias  dMAE_vs_HGB  dMAPE_pp_vs_HGB
              HGB log_rpm        all  4759 110.04    4.94 620.67 -83.58         0.00             0.00
              HGB log_rpm clean-only  4691  60.22    2.58  82.04 -55.33         0.00             0.00
HGB log_rpm +quote_signal        all  4759 148.48    7.27 624.87 -52.34        38.44             2.33
HGB log_rpm +quote_signal clean-only  4691  99.44    4.98 135.70 -23.76        39.22             2.40
        HGB log_rpm smear        all 

In [6]:
# ===== Batch C · cell 2: diagnosis only. Nothing is chosen or tuned from these Aug numbers. =====
# Clean-only: label errors (2-6x off) would swamp a bias reading.
q = tr["market_index"].quantile([.25, .5, .75]).to_numpy()          # quartile edges from TRAINING (Jan-Jul)

def add_groups(h: pd.DataFrame) -> pd.DataFrame:
    h = h[~h["is_label_error"]].copy()
    h["band"] = features.distance_band(h["distance"])               # categorical keeps 0-250 first
    h["aug_week"] = "wk" + ((h[DATE_COL].dt.day - 1) // 7 + 1).astype(str)
    h["market_q"] = pd.cut(h["market_index"], [-np.inf, *q, np.inf], labels=["Q1 low", "Q2", "Q3", "Q4 high"])
    h["err"] = h["pred"] - h[TARGET]
    h["pct"] = 100 * h["err"] / h[TARGET]
    return h

H = {k: add_groups(results[k]["holdout"]) for k in ["B1", "S1", "HGB log_rpm"]}
print("Clean-only overall bias %:", {k: round(h["pct"].mean(), 2) for k, h in H.items()})
print("market_index quartile edges (Jan-Jul):", np.round(q, 4))

for by in ["band", "equipment", "aug_week", "market_q"]:
    g = H["HGB log_rpm"].groupby(by, observed=True)
    t = pd.DataFrame({"rows": g.size()})
    for k, h in H.items():
        t[f"{k.split()[0]} bias%"] = h.groupby(by, observed=True)["pct"].mean()
    t["HGB bias$"] = g["err"].mean()
    t["HGB MAE"] = g["err"].apply(lambda e: e.abs().mean())
    print(f"\nby {by}:\n{t.round(2).to_string()}")

Clean-only overall bias %: {'B1': np.float64(1.3), 'S1': np.float64(-2.05), 'HGB log_rpm': np.float64(-2.36)}
market_index quartile edges (Jan-Jul): [1.0126 1.1608 1.2729]

by band:
              rows  B1 bias%  S1 bias%  HGB bias%  HGB bias$  HGB MAE
band                                                                 
(0, 250]       262      0.69     -2.59      -2.11      -9.51    11.06
(250, 500]     753      1.15     -2.13      -2.25     -20.43    22.63
(500, 1000]   1444      1.16     -2.17      -2.46     -40.31    43.57
(1000, 2000]  1516      1.43     -1.95      -2.37     -69.28    75.38
(2000, 4000]   716      1.65     -1.76      -2.35    -109.55   119.23

by equipment:
           rows  B1 bias%  S1 bias%  HGB bias%  HGB bias$  HGB MAE
equipment                                                         
Dry Van    2660      1.14     -2.16      -2.43     -54.36    58.13
Flatbed     850      1.49     -1.89      -2.14     -52.71    60.24
Reefer     1181      1.50     -1.93      -2.3

In [7]:
# ===== Batch C · cell 3 =====
# price_level = load's rpm / its B1 cell rpm (table from Jan-Jul training rows). 1.00 = typical.
# If Aug has a Jan/Feb-like market_index but a HIGHER price_level than Jan/Feb,
# the link between market_index and price shifts over time (the hypothesis).
table = baselines.fit_price_table(tr_fit)
months = pd.concat([tr_fit.assign(side="train (in-sample)"),
                    te[~results["HGB log_rpm"]["holdout"]["is_label_error"].to_numpy()].assign(side="Aug (holdout)")])
months["price_level"] = baselines.rpm(months) / baselines.lookup_rpm(months, table)

m_plain = models.HGBRate(target="log_rpm").fit(tr_fit)                  # same model as the candidate
months["hgb_pct"] = 100 * (m_plain.predict(months) - months[TARGET]) / months[TARGET]

by_m = months.groupby(months[DATE_COL].dt.strftime("%Y-%m")).agg(
    side=("side", "first"), rows=("price_level", "size"),
    median_market=("market_index", "median"), median_price_level=("price_level", "median"),
    hgb_bias_pct=("hgb_pct", "mean"))
print(by_m.round(4).to_string())

                      side  rows  median_market  median_price_level  hgb_bias_pct
date                                                                             
2025-01  train (in-sample)  4837         0.9326              0.9464        0.7417
2025-02  train (in-sample)  4280         0.9932              0.9613        0.7376
2025-03  train (in-sample)  4965         1.0650              0.9947       -0.9095
2025-04  train (in-sample)  4753         1.2067              1.0011        1.2914
2025-05  train (in-sample)  4851         1.3031              1.0237        0.7152
2025-06  train (in-sample)  4720         1.2749              1.0483       -1.7640
2025-07  train (in-sample)  4847         1.2122              1.0200       -0.5545
2025-08      Aug (holdout)  4691         0.9823              0.9884       -2.3603


In [8]:
# ===== Batch D · cell 1 =====
import config
for m in (config, features, baselines, validation, models, experiments):   # dependency order
    importlib.reload(m)

daily = features.daily_market_index(tr)                       # Jan-Jul, cleaned (features only)
d = pd.Timestamp("2025-07-31")
manual = daily.loc[d - pd.Timedelta(days=27): d]
func = features.trailing_market_mean(daily, pd.Series([d]), 28)[0]
print(f"28-day mean on {d.date()}: manual {manual.mean():.6f} over {len(manual)} days | function {func:.6f}")
print(f"Jan 1 (short window allowed): {features.trailing_market_mean(daily, pd.Series([daily.index.min()]), 28)[0]:.6f} "
      f"= that day's mean {daily.iloc[0]:.6f}")
try:
    features.trailing_market_mean(daily, pd.Series([pd.Timestamp("2025-09-15")]), 28)
    print("ERROR: gap was NOT refused")
except ValueError as e:
    print("Gap refused, as intended:", e)
print("Walk folds:", [f.name for f in validation.WALK_FOLDS])

28-day mean on 2025-07-31: manual 1.188029 over 28 days | function 1.188029
Jan 1 (short window allowed): 0.961224 = that day's mean 0.961224
Gap refused, as intended: 28 day(s) missing from market history, e.g. [Timestamp('2025-08-19 00:00:00'), Timestamp('2025-08-20 00:00:00'), Timestamp('2025-08-21 00:00:00')]
Walk folds: ['wf-May', 'wf-Jun', 'wf-Jul', 'wf-Aug']


In [9]:
# ===== Batch D · cell 2 =====
W = validation.WALK_FOLDS
SET = "defaults; early_stopping=False; seed 42"
walk = {}

def go(label, make, target, feats, sett, drop=True):
    t0 = time.perf_counter()
    walk[label] = experiments.run_walk(raw, W, make, target, feats, sett, drop)
    print(f"{label:40s} {time.perf_counter() - t0:5.1f}s")

go("B1", baselines.SegmentMedianRpm, "rpm (median)", "equipment+band", "median table")
go("HGB log_rpm", partial(models.HGBRate, target="log_rpm"), "log_rpm", BASE, SET)
go("HGB log_rate", partial(models.HGBRate, target="log_rate"), "log_rate", BASE, SET)
go("HGB +market_28d", partial(models.HGBRate, target="log_rpm", extra=("market_28d",)),
   "log_rpm", BASE + " + market_28d", SET)

# Reproducibility: wf-Aug must equal Batch B's decision-fold numbers
assert round(walk["B1"][-1]["scores"]["MAE"], 2) == 136.60
assert round(walk["HGB log_rpm"][-1]["scores"]["MAE"], 2) == 110.04
print("wf-Aug reproduces Batch B (B1 136.60, HGB 110.04).\n")

s1 = experiments.walk_summary(walk, ref="HGB log_rpm")
print(s1.round(2).to_string())

# The pre-fixed rule picks the feature set; nothing is chosen by eye
use_28d = bool(s1.loc["HGB +market_28d", "passes_rule"])
EXTRA = ("market_28d",) if use_28d else ()
FEATS = BASE + (" + market_28d" if use_28d else "")
CAND = "HGB +market_28d" if use_28d else "HGB log_rpm"
print(f"\nStep 2 by the rule -> candidate: {CAND}")

B1                                         0.6s
HGB log_rpm                                7.2s
HGB log_rate                               6.6s
HGB +market_28d                            6.8s
wf-Aug reproduces Batch B (B1 136.60, HGB 110.04).

                 MAE wf-May  MAE wf-Jun  MAE wf-Jul  MAE wf-Aug  mean_MAE  mean_MAPE_%  mean_clean_MAE  mean_clean_bias_%  dMAE_vs_ref  dMAPE_pp_vs_ref  wins_vs_ref  passes_rule
run                                                                                                                                                                              
B1                   165.35      212.14      148.58      136.60    165.67         7.14          113.55              -2.82        58.42             2.63            0        False
HGB log_rpm           83.59      139.84       95.50      110.04    107.24         4.51           54.32              -1.67         0.00             0.00            0        False
HGB log_rate          86.45      141.75     

In [10]:
# ===== Batch D · cell 3 =====
mk = lambda **kw: partial(models.HGBRate, target="log_rpm", extra=EXTRA, **kw)
go(CAND + " | errors KEPT", mk(), "log_rpm", FEATS, SET, drop=False)
go(CAND + " | i lr0.05 x400", mk(learning_rate=0.05, max_iter=400), "log_rpm", FEATS, SET + "; lr=0.05 max_iter=400")
go(CAND + " | ii leaves63", mk(max_leaf_nodes=63), "log_rpm", FEATS, SET + "; max_leaf_nodes=63")
go(CAND + " | iii minleaf100", mk(min_samples_leaf=100), "log_rpm", FEATS, SET + "; min_samples_leaf=100")

keys = [CAND] + [k for k in walk if k.startswith(CAND + " |")]
s2 = experiments.walk_summary({k: walk[k] for k in keys}, ref=CAND)
print(s2.round(2).to_string())
print("\nLog rows:", len(pd.read_csv(experiments.LOG_PATH)), "(expected 6 + 16 + 16 = 38)")

HGB +market_28d | errors KEPT              6.5s
HGB +market_28d | i lr0.05 x400           20.2s
HGB +market_28d | ii leaves63              9.1s
HGB +market_28d | iii minleaf100           5.8s
                                  MAE wf-May  MAE wf-Jun  MAE wf-Jul  MAE wf-Aug  mean_MAE  mean_MAPE_%  mean_clean_MAE  mean_clean_bias_%  dMAE_vs_ref  dMAPE_pp_vs_ref  wins_vs_ref  passes_rule
run                                                                                                                                                                                               
HGB +market_28d                        82.46      132.50       92.46      100.44    101.96         4.30           49.03              -1.23         0.00             0.00            0        False
HGB +market_28d | errors KEPT         129.22      151.38      106.34      107.70    123.66         5.28           70.98              -1.66        21.70             0.98            0        False
HGB +market_28d | i lr0.05 x

In [11]:
# ===== Batch E: Sep-Oct main holdout (D20). Looked at ONCE, chosen model only. Nothing is tuned after this. =====
FINAL_SET = SET + "; lr=0.05 max_iter=400"
chosen = partial(models.HGBRate, target="log_rpm", extra=("market_28d",), learning_rate=0.05, max_iter=400)

main = {"B1": validation.run_fold(raw, validation.MAIN_FOLD, baselines.SegmentMedianRpm)}
assert round(main["B1"]["scores"]["MAE"], 2) == 140.53          # bar reproduces Phase 3
main["HGB"] = validation.run_fold(raw, validation.MAIN_FOLD, chosen)
experiments.log_experiments([
    experiments.experiment_row(main["B1"], "rpm (median)", "equipment+band", "median table"),
    experiments.experiment_row(main["HGB"], "log_rpm", BASE + " + market_28d", FINAL_SET)])

r = main["HGB"]
print(f"{r['model']}\n cleaning_rows {r['cleaning_rows']:,} | dropped {r['dropped']} | fit_rows {r['fit_rows']:,} | "
      f"test_rows {r['test_rows']:,} | train_last_date {r['train_last_date']}\n")

# 1) Headline: all rows + clean-only, vs B1
tab = validation.results_table([main["B1"], main["HGB"]])
b1 = tab[tab["model"].str.startswith("B1")].set_index("scope")
tab["dMAE_vs_B1"] = tab["MAE"] - tab["scope"].map(b1["MAE"])
tab["dMAPE_pp_vs_B1"] = tab["MAPE_%"] - tab["scope"].map(b1["MAPE_%"])
print(tab.drop(columns=["fold", "train_label_errors"]).round(2).to_string(index=False))

h = r["holdout"]; hc = h[~h["is_label_error"]]
print(f"\nClean-only bias %: HGB {100 * ((hc['pred'] - hc[TARGET]) / hc[TARGET]).mean():.2f}")

# 2) Breakdowns side by side (all rows, plus month clean-only)
def side_by_side(by, clean=False):
    pick = lambda x: x[~x["is_label_error"]] if clean else x
    b = validation.breakdown(pick(main["B1"]["holdout"]), by)
    c = validation.breakdown(pick(main["HGB"]["holdout"]), by)
    t = b[[by, "rows", "label_errors", "MAE", "MAPE_%", "bias"]].merge(
        c[[by, "MAE", "MAPE_%", "bias"]], on=by, suffixes=("_B1", "_HGB"))
    print(f"\nby {by}{' (clean-only)' if clean else ''}:\n{t.round(2).to_string(index=False)}")

for by in ["equipment", "band", "month"]:
    side_by_side(by)
side_by_side("month", clean=True)

# 3) Extrapolation check (P1): Sep-Oct rows below the lowest market_index seen in Jan-Aug training
lo = main_tr_raw["market_index"].min()      # raw minimum (equal to the cleaned min of 0.7325, since filling uses same-date means)
below = h["market_index"] < lo
for name, part in [("below training min", h[below]), ("inside range", h[~below])]:
    s = validation.score(part[TARGET], part["pred"])
    print(f"\n{name}: rows {len(part)} | MAE {s['MAE']:.2f} | MAPE {s['MAPE_%']:.2f}% | bias {s['bias']:.2f}")
print("\nLog rows:", len(pd.read_csv(experiments.LOG_PATH)), "(expected 40)")

HGB log_rpm +market_28d learning_rate=0.05 max_iter=400
 cleaning_rows 38,477 | dropped 533 | fit_rows 37,944 | test_rows 9,523 | train_last_date 2025-08-31

                                                  model      scope  rows    MAE  MAPE_%   RMSE    bias  dMAE_vs_B1  dMAPE_pp_vs_B1
                      B1 median rpm by equipment x band        all  9523 140.53    6.40 638.82  -43.12        0.00            0.00
                      B1 median rpm by equipment x band clean-only  9379  85.01    3.90 116.40   -9.55        0.00            0.00
HGB log_rpm +market_28d learning_rate=0.05 max_iter=400        all  9523 188.31    7.91 657.54 -167.04       47.78            1.51
HGB log_rpm +market_28d learning_rate=0.05 max_iter=400 clean-only  9379 133.58    5.57 167.33 -133.45       48.57            1.68

Clean-only bias %: HGB -5.57

by equipment:
equipment  rows  label_errors  MAE_B1  MAPE_%_B1  bias_B1  MAE_HGB  MAPE_%_HGB  bias_HGB
  Dry Van  5360            85  139.26       6.20   -4

In [12]:
# ===== Batch E2 =====
importlib.reload(models)

# 1) Did the market -> price link break? Monthly, Jan-Oct (diagnosis; Sep-Oct labels already seen once)
main_tr = features.clean(main_tr_raw, L_main)
mtr_fit, _ = baselines.drop_label_errors(main_tr)
table_m = baselines.fit_price_table(mtr_fit)
full = windows["full-train Jan-Oct"]
ok = full[~baselines.flag_label_errors(full, table_m)["is_error"].to_numpy()].copy()
ok["price_level"] = baselines.rpm(ok) / baselines.lookup_rpm(ok, table_m)
ok["market_28d"] = features.trailing_market_mean(features.daily_market_index(full), ok[DATE_COL], 28)
print(ok.groupby(ok[DATE_COL].dt.strftime("%Y-%m")).agg(
    median_market=("market_index", "median"), median_market_28d=("market_28d", "median"),
    median_price_level=("price_level", "median")).round(4).to_string())

# 2) What did the failed model learn? Average Sep-Oct prediction if market were set to v
m_main = chosen().fit(mtr_fit)
so = features.clean(so_raw, L_main)
assert np.allclose(m_main.predict(so), main["HGB"]["holdout"]["pred"].to_numpy())   # same model as Batch E
X = models.build_features(so, m_main.extra, m_main._history(so))[m_main.columns_]
ref_p = None
print("\nmarket v -> mean predicted $ (ratio to v=1.0):")
for v in [0.8, 0.9, 1.0, 1.1, 1.2, 1.3]:
    Xg = X.copy(); Xg["market_index"] = v; Xg["market_28d"] = v
    p = np.exp(m_main.model_.predict(Xg) + m_main._offset(so)).mean()
    ref_p = p if v == 1.0 else ref_p
    print(f"  {v:.1f}: ${p:,.0f}", end="")
print(f"\n  (ratio 0.9 vs 1.0 shown after loop) actual Sep-Oct mean price ${so[TARGET].mean():,.0f}")

# 3) Walk-forward (folds end by Aug) for options a and b, vs B1 and the failed model
go("HGB no-market", partial(models.HGBRate, target="log_rpm", drop=("market_index",),
   learning_rate=0.05, max_iter=400), "log_rpm", BASE + " minus market_index", FINAL_SET)
go("HGB market_rel only", partial(models.HGBRate, target="log_rpm", extra=("market_rel",), drop=("market_index",),
   learning_rate=0.05, max_iter=400), "log_rpm", BASE + " minus market_index + market_rel", FINAL_SET)
s3 = experiments.walk_summary({k: walk[k] for k in
     ["B1", CAND + " | i lr0.05 x400", "HGB no-market", "HGB market_rel only"]}, ref="B1")
print("\n", s3.round(2).to_string())
print("\nLog rows:", len(pd.read_csv(experiments.LOG_PATH)), "(expected 48)")

         median_market  median_market_28d  median_price_level
date                                                         
2025-01         0.9326             0.9148              0.9481
2025-02         0.9932             0.9586              0.9631
2025-03         1.0650             1.0429              0.9963
2025-04         1.2067             1.1230              1.0028
2025-05         1.3031             1.2864              1.0255
2025-06         1.2749             1.2963              1.0498
2025-07         1.2122             1.2610              1.0215
2025-08         0.9823             1.0506              0.9900
2025-09         0.8879             0.9349              1.0060
2025-10         0.9576             0.9191              1.0086

market v -> mean predicted $ (ratio to v=1.0):
  0.8: $2,186  0.9: $2,215  1.0: $2,288  1.1: $2,302  1.2: $2,368  1.3: $2,432
  (ratio 0.9 vs 1.0 shown after loop) actual Sep-Oct mean price $2,392
HGB no-market                             26.8s
HGB market

In [13]:
# ===== Batch E3 =====
# 1) Decision-3 tests that were missed, on the folds, vs the new base (no market_index).
#    Pre-registered: an extra is adopted only if it passes the rule; if both pass, only the better one.
NM = dict(target="log_rpm", drop=("market_index",), learning_rate=0.05, max_iter=400)
go("HGB no-market +weekday", partial(models.HGBRate, extra=("weekday",), **NM),
   "log_rpm", BASE + " minus market_index + weekday", FINAL_SET)
go("HGB no-market +circuity", partial(models.HGBRate, extra=("circuity",), **NM),
   "log_rpm", BASE + " minus market_index + circuity", FINAL_SET)
s4 = experiments.walk_summary({k: walk[k] for k in
     ["HGB no-market", "HGB no-market +weekday", "HGB no-market +circuity"]}, ref="HGB no-market")
print(s4.round(2).to_string())

passing = s4[s4["passes_rule"]].sort_values("mean_MAE")
FINAL_EXTRA = () if passing.empty else {"HGB no-market +weekday": ("weekday",),
                                        "HGB no-market +circuity": ("circuity",)}[passing.index[0]]
FINAL_FEATS = BASE + " minus market_index" + "".join(f" + {e}" for e in FINAL_EXTRA)
final_make = partial(models.HGBRate, extra=FINAL_EXTRA, **NM)
print(f"\nFinal features by the rule: {FINAL_FEATS}")

# 2) SECOND LOOK at Sep-Oct (disclosed). Pre-registered pass: MAE < 140.53 AND MAPE <= 6.50 (all rows)
main["HGB2"] = validation.run_fold(raw, validation.MAIN_FOLD, final_make)
experiments.log_experiments([experiments.experiment_row(
    main["HGB2"], "log_rpm", FINAL_FEATS, FINAL_SET + "; SECOND LOOK at Sep-Oct after failed market model")])
r = main["HGB2"]
print(f"\n{r['model']} | cleaning {r['cleaning_rows']:,} | dropped {r['dropped']} | fit {r['fit_rows']:,} | test {r['test_rows']:,}")

tab = validation.results_table([main["B1"], main["HGB2"]])
b1 = tab[tab["model"].str.startswith("B1")].set_index("scope")
tab["dMAE_vs_B1"] = tab["MAE"] - tab["scope"].map(b1["MAE"])
tab["dMAPE_pp_vs_B1"] = tab["MAPE_%"] - tab["scope"].map(b1["MAPE_%"])
print(tab.drop(columns=["fold", "train_label_errors"]).round(2).to_string(index=False))
s = r["scores"]
print("\nVERDICT:", "PASSES the pre-registered second look" if (s["MAE"] < 140.53 and s["MAPE_%"] <= 6.50)
      else "FAILS -> B1 is the submission model")

hc = r["holdout"][~r["holdout"]["is_label_error"]]
print(f"Clean-only bias %: {100 * ((hc['pred'] - hc[TARGET]) / hc[TARGET]).mean():.2f}")

def side_by_side(by, clean=False, key="HGB2"):
    pick = lambda x: x[~x["is_label_error"]] if clean else x
    b = validation.breakdown(pick(main["B1"]["holdout"]), by)
    c = validation.breakdown(pick(main[key]["holdout"]), by)
    t = b[[by, "rows", "MAE", "MAPE_%", "bias"]].merge(c[[by, "MAE", "MAPE_%", "bias"]], on=by, suffixes=("_B1", "_HGB"))
    print(f"\nby {by}{' (clean-only)' if clean else ''}:\n{t.round(2).to_string(index=False)}")

for by in ["equipment", "band", "month"]:
    side_by_side(by)
side_by_side("month", clean=True)
print("\nLog rows:", len(pd.read_csv(experiments.LOG_PATH)), "(expected 57)")

HGB no-market +weekday                    37.9s
HGB no-market +circuity                   21.8s
                         MAE wf-May  MAE wf-Jun  MAE wf-Jul  MAE wf-Aug  mean_MAE  mean_MAPE_%  mean_clean_MAE  mean_clean_bias_%  dMAE_vs_ref  dMAPE_pp_vs_ref  wins_vs_ref  passes_rule
run                                                                                                                                                                                      
HGB no-market                157.23      210.72      122.10       93.10    145.79         5.97           93.41               -2.9         0.00             0.00            0        False
HGB no-market +weekday       155.81      212.15      119.38       89.99    144.33         5.92           91.94               -2.9        -1.45            -0.06            3         True
HGB no-market +circuity      157.20      210.36      122.41       93.38    145.84         5.97           93.48               -2.9         0.05             0.00 

In [14]:
# ===== Batch F =====
from config import SEED, DECEMBER_PATH
importlib.reload(models)
final_make = partial(models.HGBRate, target="log_rpm", extra=("weekday",), drop=("market_index",),
                     learning_rate=0.05, max_iter=400)
print("Final model name:", final_make().name)                    # must now show -market_index
FSET = FINAL_SET

# --- 1) Random vs time (D26): same pool, test size 9,523, 3 seeds. Comparison only, chooses nothing.
rows_f, rv = [], {}
for sd in [42, 1, 2]:
    tr_r, te_r = validation.random_split(raw, 9523, seed=sd)
    rv[sd] = validation.run_split(tr_r, te_r, f"random s{sd}", final_make)
    rows_f.append(experiments.experiment_row(rv[sd], "log_rpm", FINAL_FEATS, FSET + "; comparison only"))
cmp_rt = validation.results_table([main["HGB2"]] + list(rv.values()))
print("\n1) Random vs time:\n", cmp_rt.drop(columns=["model", "train_label_errors"]).round(2).to_string(index=False))

# --- 2) Unseen cities (D29) on the Aug fold: same Aug rows, model with vs without those cities in training.
tr_raw_d, te_raw_d = validation.time_split(raw, validation.DECISION_FOLD)
cities = sorted(set(tr_raw_d["pickup"]) | set(tr_raw_d["delivery"]))
held = sorted(np.random.default_rng(SEED).choice(cities, 8, replace=False))
touch = lambda d: d["pickup"].isin(held) | d["delivery"].isin(held)
te_sub = te_raw_d[touch(te_raw_d)]
print(f"\n2) Held-out cities: {held}\n   training rows removed: {int(touch(tr_raw_d).sum()):,} of {len(tr_raw_d):,} | Aug rows scored: {len(te_sub):,}")
uc = {"with cities": validation.run_split(tr_raw_d, te_sub, "wf-Aug held-out cities: WITH in training", final_make),
      "without cities": validation.run_split(tr_raw_d[~touch(tr_raw_d)], te_sub, "wf-Aug held-out cities: WITHOUT in training", final_make),
      "B1 (no location)": validation.run_split(tr_raw_d, te_sub, "wf-Aug held-out cities: B1", baselines.SegmentMedianRpm)}
for k, r in uc.items():
    rows_f.append(experiments.experiment_row(r, "log_rpm" if "B1" not in k else "rpm (median)",
                  FINAL_FEATS if "B1" not in k else "equipment+band", FSET + f"; held-out cities {held}"))
print(validation.results_table(list(uc.values())).drop(columns=["model", "train_label_errors"]).round(2).to_string(index=False))
experiments.log_experiments(rows_f)

# --- 3) Permutation importance (decision 12, hand-written), Aug fold, all rows, 3 repeats.
#        Shuffle one feature group in the model's inputs; MAE increase = how much the model relies on it.
#        Distance is shuffled as a FEATURE only; the per-mile conversion (x distance) is kept.
m = final_make().fit(tr_fit)
X = models.build_features(te, m.extra, m._history(te)).drop(columns=list(m.drop))[m.columns_]
off, y = m._offset(te), te[TARGET].to_numpy(dtype=float)
base_mae = validation.mae(y, np.exp(m.model_.predict(X) + off))
assert round(base_mae, 2) == round(walk["HGB no-market +weekday"][-1]["scores"]["MAE"], 2)   # = wf-Aug result
groups = {"distance": ["distance"], "weight": ["weight"], "pickup location": ["pickup_lat", "pickup_lon"],
          "delivery location": ["delivery_lat", "delivery_lon"],
          "equipment": [c for c in m.columns_ if c.startswith("eq_")], "weekday": ["weekday"]}
rng = np.random.default_rng(SEED)
imp = []
for g, cols in groups.items():
    inc = []
    for _ in range(3):
        Xp = X.copy(); Xp[cols] = Xp[cols].to_numpy()[rng.permutation(len(Xp))]
        inc.append(validation.mae(y, np.exp(m.model_.predict(Xp) + off)) - base_mae)
    imp.append({"feature": g, "MAE_increase_$": np.mean(inc), "sd": np.std(inc)})
print(f"\n3) Permutation importance (Aug, base MAE {base_mae:.2f}):\n",
      pd.DataFrame(imp).sort_values("MAE_increase_$", ascending=False).round(2).to_string(index=False))

# --- 4) December + validation look: final model trained on ALL Jan-Oct (as in Phase 6). No files written.
L_all = features.fit_cleaning(raw)
all_fit, _ = baselines.drop_label_errors(features.clean(raw, L_all))
mf = final_make().fit(all_fit)
v_clean = features.clean(vraw, L_all)
pv = mf.predict(v_clean)
print(f"\n4) Validation: n={len(pv):,} finite={np.isfinite(pv).all()} >0={(pv > 0).all()} | "
      f"median ${np.median(pv):,.0f} (Sep-Oct actual median ${so[TARGET].median():,.0f})")
dec = features.clean(pd.read_csv(DECEMBER_PATH), L_all, market_by_date=features.daily_market_index(v_clean))
dec["pred"] = mf.predict(dec)
dec["weekday"] = dec[DATE_COL].dt.day_name()
print(f"   December: min ${dec['pred'].min():.2f}, max ${dec['pred'].max():.2f}, distinct values {dec['pred'].round(2).nunique()} "
      f"(sanity band 750-950; B1 flat 836.89)")
print(dec.groupby("weekday", sort=False)["pred"].first().round(2).to_string())
print("\nLog rows:", len(pd.read_csv(experiments.LOG_PATH)), "(expected 63)")

Final model name: HGB log_rpm -market_index +weekday learning_rate=0.05 max_iter=400

1) Random vs time:
       fold      scope  rows    MAE  MAPE_%   RMSE   bias
      main        all  9523  98.34    4.33 631.13 -51.02
      main clean-only  9379  42.31    1.81  62.54 -17.34
random s42        all  9523 112.43    5.07 533.78 -27.13
random s42 clean-only  9394  67.69    2.82  98.85  -1.59
 random s1        all  9523 119.17    5.28 621.67 -29.59
 random s1 clean-only  9388  67.49    2.82  98.40  -0.20
 random s2        all  9523 107.73    4.95 555.80 -20.23
 random s2 clean-only  9412  68.12    2.85  97.94  -1.62

2) Held-out cities: [np.str_('Bakersfield'), np.str_('Hartford'), np.str_('Memphis'), np.str_('New York'), np.str_('San Antonio'), np.str_('Toledo'), np.str_('Tulsa'), np.str_('Washington')]
   training rows removed: 8,820 of 33,718 | Aug rows scored: 1,226
                                       fold      scope  rows    MAE  MAPE_%   RMSE  bias
   wf-Aug held-out cities: WITH i